# Chapter 06. Streamlit으로 도서 분류·추천 앱 완성하기

앞 Chapter까지 도서 제목을 TF-IDF로 변환하고, Naive Bayes로 분야를 분류하고,
코사인 유사도로 비슷한 도서를 추천했다.

이번 Chapter에서는 **새로운 머신러닝 알고리즘을 배우지 않는다.**
앞에서 만든 두 기능을 Streamlit 화면에 연결해 사용자가 직접 실행할 수 있는 작은 웹 앱으로 완성한다.

```
[도서 제목 입력]  -> TF-IDF transform -> Naive Bayes        -> 예상 분야
[기존 도서 선택]  -> TF-IDF            -> Cosine Similarity  -> 유사 도서 Top 5
```

> **핵심 목표:** Notebook의 분석 기능을 **사용자 입력 -> 분석 -> 결과 출력**의 앱 흐름으로 연결한다.

## 이 Notebook과 app.py의 역할

| 파일 | 역할 |
|---|---|
| `app.py` | **실제 Streamlit 앱.** 터미널에서 `python -m streamlit run app.py`로 실행한다 |
| `chapter06.ipynb` | 앱의 각 부분이 **무엇을 하는지 설명하고, 로직이 맞는지 검증**한다 |

Streamlit 코드(`st.title`, `st.button` 등)는 **Notebook 셀에서 실행하면 화면이 나오지 않는다.**
`missing ScriptRunContext` 경고만 뜬다. Streamlit은 `.py` 파일을 `streamlit run`으로 실행해야 브라우저에 앱이 뜬다.

그래서 이 Notebook에서는 앱 코드를 **설명용으로 보여주고**, 실행 셀에서는
Streamlit을 뺀 같은 로직을 돌리거나, `AppTest`로 **실제 app.py를 자동으로 눌러보며** 검증한다.

---
# 1. 학습 목표

- Streamlit 앱을 실행할 수 있다.
- CSV 데이터를 앱에서 불러올 수 있다.
- `st.cache_data`와 `st.cache_resource`의 역할을 설명할 수 있다.
- 제목 입력창을 Naive Bayes 분류 모델과 연결할 수 있다.
- 도서 선택창을 코사인 유사도 추천 함수와 연결할 수 있다.
- 분류와 추천 기능을 하나의 `app.py`로 통합할 수 있다.
- 브라우저에서 실제 입력으로 결과를 검증할 수 있다.
- 앱 결과의 한계를 설명할 수 있다.

## 전체 흐름

```
Streamlit 실행 -> CSV 로드 -> 분류 모델 준비 -> text_input -> 분야 예측
-> 추천 데이터 준비 -> selectbox -> Top 5 추천 -> app.py 통합 -> 브라우저 검증
```

### 작업 폴더를 맞춘다

이 저장소의 `.vscode/settings.json`에 `"jupyter.notebookFileRoot": "${workspaceFolder}"`가 있어서,
Notebook을 실행하면 작업 폴더가 **저장소 루트**가 된다.
아래 셀에서 작업 폴더를 `book-text-ml`로 맞춘 뒤 진행한다.

> **앱을 실행할 때도 같은 문제가 생긴다.** 터미널이 저장소 루트에 있으면
> `app.py`가 CSV를 찾지 못한다. 반드시 `cd notebooks/book-text-ml`로 이동한 뒤 실행한다.

## 프롬프트 예시 (이 확인 셀)

```
VS Code Jupyter 설정 때문에 작업 폴더가 저장소 루트가 됩니다.
노트북이 들어 있는 book-text-ml 폴더를 찾아 그쪽으로 이동하고,
작업 폴더와 app.py, 데이터 파일이 있는지 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [1]:
import os
from pathlib import Path

# 노트북이 있는 book-text-ml 폴더를 찾아 그쪽으로 이동한다.
if Path.cwd().name != "book-text-ml":
    for candidate in [Path("book-text-ml"), *Path(".").glob("*/book-text-ml")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())
print("app.py 있음     :", Path("app.py").exists())
print("데이터 파일 있음:", Path("book_bestseller_clean.csv").exists())

작업 폴더: C:\dev\llm-data-analysis-course\notebooks\book-text-ml
app.py 있음     : True
데이터 파일 있음: True


---
# 실습 1. Streamlit 설치와 최소 앱 실행

## 해야 할 일

필요한 패키지를 설치하고 버전을 확인한다.

```
python -m pip install streamlit pandas scikit-learn
python -m streamlit --version
```

가장 작은 `app.py`를 만들어 실행해 본다.

```python
import streamlit as st

st.title("교보문고 베스트셀러 텍스트 분석 앱")
st.write("도서 분야 분류와 유사 도서 추천 기능을 실습합니다.")
```

```
cd notebooks/book-text-ml
python -m streamlit run app.py
```

브라우저에 제목과 설명이 보이면 Streamlit 실행 환경은 정상이다.
앱을 중지할 때는 터미널에서 `Ctrl + C`를 누른다.

## 프롬프트 예시

```
Windows와 VS Code에서 Streamlit을 처음 써보려고 합니다.
제목 '교보문고 베스트셀러 텍스트 분석 앱'과
설명 한 줄을 보여주는 가장 작은 app.py를 만들어 주세요.
그리고 이 파일을 실행하는 터미널 명령어도 알려주세요.
불필요한 설정이나 레이아웃 코드는 넣지 말아 주세요.
3줄 이내로 짧게 작성해 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
Notebook에서 Streamlit이 설치되어 있는지 버전을 확인하고,
같은 폴더의 app.py가 몇 줄인지 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [2]:
import streamlit

print("Streamlit 버전:", streamlit.__version__)
print("app.py 줄 수:", len(Path("app.py").read_text(encoding="utf-8").splitlines()))

Streamlit 버전: 1.63.0
app.py 줄 수: 164


---
# 실습 2. CSV 데이터 불러오기

## 해야 할 일

`app.py`와 전처리 CSV를 같은 폴더에 둔다. 필수 컬럼은 `상품명`, `분야` 두 개다.

앱 UI보다 **먼저 데이터가 정상적으로 읽히는지 확인**하는 것이 중요하다.

### app.py 안의 코드

```python
DATA_PATH = "book_bestseller_clean.csv"

@st.cache_data
def load_data():
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    required_columns = ["상품명", "분야"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")
    df["상품명"] = df["상품명"].fillna("").astype(str).str.strip()
    df["분야"] = df["분야"].fillna("미분류").astype(str).str.strip()
    df = df[df["상품명"] != ""].reset_index(drop=True)
    return df
```

## 프롬프트 예시

```
Streamlit 앱에서 book_bestseller_clean.csv를 불러오는 load_data() 함수를 만들고 싶습니다.
1. utf-8-sig로 읽고
2. 상품명, 분야 컬럼이 없으면 ValueError를 발생시키고
3. 상품명은 빈 문자열, 분야는 '미분류'로 결측치를 채우고 앞뒤 공백을 지운 뒤
4. 빈 제목을 빼고 index를 다시 매겨 돌려주세요.
st.cache_data로 결과를 재사용하게 해주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

### Notebook에서 같은 로직 확인

아래는 `app.py`의 `load_data()`와 **같은 로직**이다. 캐시 데코레이터만 뺐다.

## 프롬프트 예시 (이 확인 셀)

```
app.py의 load_data()와 같은 로직을 Notebook에서 확인하고 싶습니다.
st.cache_data만 빼고 같은 함수를 만들어 실행한 뒤,
데이터 크기, 컬럼 목록, 미분류 도서 수를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [3]:
import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"

def load_data():
    df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
    required_columns = ["상품명", "분야"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")
    df["상품명"] = df["상품명"].fillna("").astype(str).str.strip()
    df["분야"] = df["분야"].fillna("미분류").astype(str).str.strip()
    df = df[df["상품명"] != ""].reset_index(drop=True)
    return df

df = load_data()
print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())
print("미분류 도서 수:", int((df["분야"] == "미분류").sum()))

데이터 크기: (986, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
미분류 도서 수: 64


---
# 실습 3. Streamlit 캐시 이해하기

Streamlit은 사용자가 **버튼을 누르거나 값을 바꿀 때마다 스크립트 전체를 처음부터 다시 실행**한다.
매번 CSV를 읽고 모델을 다시 학습하면 불필요한 작업이 반복된다.

| 캐시 | 역할 | 이번 앱에서 |
|---|---|---|
| `st.cache_data` | 데이터 로딩 결과 재사용 | `load_data()` |
| `st.cache_resource` | Vectorizer, 모델 등 재사용 | `train_classifier()`, `build_recommender()` |

> 자주 다시 만들 필요가 없는 결과를 기억해 두고 재사용한다.

## 프롬프트 예시 (이 확인 셀)

```
Streamlit 캐시가 왜 필요한지 확인하고 싶습니다.
CSV 로드와 TF-IDF + MultinomialNB 학습을 한 번 하는 데 걸리는 시간을
밀리초 단위로 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [4]:
# 캐시가 없으면 버튼을 누를 때마다 이만큼의 시간이 반복된다
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

start = time.perf_counter()
_df = load_data()
_train = _df[_df["분야"] != "미분류"]
_vec = TfidfVectorizer()
MultinomialNB().fit(_vec.fit_transform(_train["상품명"]), _train["분야"])
elapsed = time.perf_counter() - start

print(f"CSV 로드 + 모델 학습 한 번: {elapsed * 1000:.0f} ms")
print("캐시를 쓰면 이 작업은 앱이 처음 뜰 때 한 번만 한다.")

CSV 로드 + 모델 학습 한 번: 27 ms
캐시를 쓰면 이 작업은 앱이 처음 뜰 때 한 번만 한다.


---
# 실습 4. 분류 모델 준비하기

## 해야 할 일

Chapter 04에서는 모델을 **평가**하려고 train/test를 나눴다.
이번 Chapter는 평가가 끝난 뒤 **시연용 앱**을 만드는 단계이므로
사용 가능한 라벨 데이터를 모두 이용해 최종 모델을 학습한다.

> 같은 데이터로 다시 정확도를 계산해서 성능이라고 보고하는 것은 아니다.
> 성능은 Chapter 04의 test 결과(0.4130)가 기준이다.

`미분류`는 실제 분야가 아니므로 학습에서 뺀다.

### app.py 안의 코드

```python
@st.cache_resource
def train_classifier():
    df = load_data()
    train_df = df[df["분야"] != "미분류"].copy()
    if train_df["분야"].nunique() < 2:
        raise ValueError("서로 다른 분야가 2개 이상 필요합니다.")
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(train_df["상품명"])
    y = train_df["분야"]
    model = MultinomialNB()
    model.fit(X, y)
    return vectorizer, model
```

## 프롬프트 예시

```
Streamlit 앱에서 쓸 분야 분류 모델을 준비하는 train_classifier() 함수를 만들고 싶습니다.
load_data()로 데이터를 불러와 분야가 '미분류'인 행은 빼고,
분야가 2개 미만이면 ValueError를 발생시키고,
상품명으로 TfidfVectorizer를 fit_transform한 뒤 MultinomialNB를 학습해서
vectorizer와 model을 함께 돌려주세요.
st.cache_resource로 한 번만 학습하게 해주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
app.py의 train_classifier()와 같은 로직을 Notebook에서 확인하고 싶습니다.
st.cache_resource만 빼고 같은 함수를 만들어 실행한 뒤,
학습한 분야 수와 단어 수를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# app.py의 train_classifier()와 같은 로직 (캐시만 제외)
def train_classifier():
    df = load_data()
    train_df = df[df["분야"] != "미분류"].copy()
    if train_df["분야"].nunique() < 2:
        raise ValueError("서로 다른 분야가 2개 이상 필요합니다.")
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(train_df["상품명"])
    y = train_df["분야"]
    model = MultinomialNB()
    model.fit(X, y)
    return vectorizer, model

classifier_vectorizer, classifier_model = train_classifier()

print("학습한 분야 수:", len(classifier_model.classes_))
print("사용한 단어 수:", len(classifier_vectorizer.vocabulary_))

학습한 분야 수: 37
사용한 단어 수: 2144


Chapter 04와 달리 **1권뿐인 분야도 제외하지 않았다.**
train/test로 나누지 않으니 `stratify` 오류가 생기지 않기 때문이다.
그래서 학습 도서는 922권, 분야는 37개가 된다.

---
# 실습 5. 제목 입력 UI와 분야 예측 연결

## 해야 할 일

사용자가 직접 제목을 입력하도록 만든다.

**새 제목에서는 다시 `fit_transform()`하지 않는다.**

```
학습        -> fit_transform()
새 제목 예측 -> transform()
```

이미 학습된 단어 공간을 그대로 사용해야 하기 때문이다.

### app.py 안의 코드

```python
classifier_vectorizer, classifier_model = train_classifier()

st.header("1. 도서 분야 예측")
user_title = st.text_input(
    "도서 제목을 입력하세요",
    placeholder="예: 처음 배우는 파이썬 데이터 분석",
)
if st.button("분야 예측"):
    clean_title = user_title.strip()
    if not clean_title:
        st.warning("도서 제목을 입력해 주세요.")
    else:
        title_vector = classifier_vectorizer.transform([clean_title])
        predicted_category = classifier_model.predict(title_vector)[0]
        st.success(f"예상 분야: {predicted_category}")
```

## 프롬프트 예시

```
Streamlit 앱에 도서 분야 예측 화면을 만들고 싶습니다.
이미 학습된 classifier_vectorizer와 classifier_model이 있습니다.
'1. 도서 분야 예측' 제목, 제목 입력창, '분야 예측' 버튼을 만들고
버튼을 누르면 입력값의 앞뒤 공백을 지운 뒤
비어 있으면 st.warning으로 안내하고,
아니면 transform()만 써서 예측한 분야를 st.success로 보여주세요.
새 제목에 fit_transform을 쓰면 안 됩니다.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
학습된 classifier_vectorizer와 classifier_model이 있습니다.
앱의 '분야 예측' 버튼과 같은 로직으로 여러 제목을 예측해 보고 싶습니다.
빈 제목은 안내 문구를 출력하고, 나머지는 transform()만 써서 예측 분야를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [6]:
# 버튼을 눌렀을 때와 같은 로직을 여러 제목으로 돌려본다
test_titles = [
    "",
    "처음 배우는 파이썬 데이터 분석",
    "인공지능 시대의 데이터 분석",
    "처음 배우는 주식 투자",
    "2027 해커스 토익 기출",
    "흔한남매 과학 탐험대",
]

for title in test_titles:
    clean_title = title.strip()
    if not clean_title:
        print("[빈 제목]                      -> 도서 제목을 입력해 주세요.")
        continue
    title_vector = classifier_vectorizer.transform([clean_title])
    predicted = classifier_model.predict(title_vector)[0]
    print(f"{clean_title:28} -> 예상 분야: {predicted}")

[빈 제목]                      -> 도서 제목을 입력해 주세요.
처음 배우는 파이썬 데이터 분석            -> 예상 분야: 어린이(초등)
인공지능 시대의 데이터 분석              -> 예상 분야: 소설
처음 배우는 주식 투자                 -> 예상 분야: 경제/경영
2027 해커스 토익 기출               -> 예상 분야: 외국어
흔한남매 과학 탐험대                  -> 예상 분야: 어린이(초등)


---
# 실습 6. 분류 결과 해석하기

## 해야 할 일

블로그 예시는 `인공지능 시대의 데이터 분석 -> 컴퓨터/IT`였는데,
실제로 돌려보니 **소설**이 나왔다. `처음 배우는 파이썬 데이터 분석`은 **어린이(초등)** 이 나왔다.

결과가 이상하면 바로 앱 오류라고 판단하지 말고 **왜 그렇게 예측했는지** 확인한다.

## 프롬프트 예시 (이 확인 셀)

```
예측 결과가 이상한 제목이 있습니다.
각 제목마다 TF-IDF 단어 사전에 있던 단어와,
predict_proba로 확률이 높은 분야 3개를 출력해서 이유를 확인하고 싶습니다.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [7]:
feature_names = classifier_vectorizer.get_feature_names_out()

for title in test_titles[1:]:
    vector = classifier_vectorizer.transform([title])
    probs = classifier_model.predict_proba(vector)[0]
    top3 = probs.argsort()[::-1][:3]

    print(f"[{title}]")
    print("   사전에 있던 단어:", feature_names[vector.indices].tolist() or "없음")
    print("   확률 상위 3개  :", [(classifier_model.classes_[k], round(probs[k], 3)) for k in top3])
    print()

[처음 배우는 파이썬 데이터 분석]
   사전에 있던 단어: ['처음']
   확률 상위 3개  : [(np.str_('어린이(초등)'), np.float64(0.245)), (np.str_('소설'), np.float64(0.137)), (np.str_('인문'), np.float64(0.113))]

[인공지능 시대의 데이터 분석]
   사전에 있던 단어: 없음
   확률 상위 3개  : [(np.str_('소설'), np.float64(0.193)), (np.str_('인문'), np.float64(0.108)), (np.str_('어린이(초등)'), np.float64(0.091))]

[처음 배우는 주식 투자]
   사전에 있던 단어: ['주식', '처음', '투자']
   확률 상위 3개  : [(np.str_('경제/경영'), np.float64(0.191)), (np.str_('소설'), np.float64(0.137)), (np.str_('어린이(초등)'), np.float64(0.134))]

[2027 해커스 토익 기출]
   사전에 있던 단어: ['2027', '기출', '토익', '해커스']
   확률 상위 3개  : [(np.str_('외국어'), np.float64(0.309)), (np.str_('취업/수험서'), np.float64(0.288)), (np.str_('소설'), np.float64(0.076))]

[흔한남매 과학 탐험대]
   사전에 있던 단어: ['과학', '탐험대', '흔한남매']
   확률 상위 3개  : [(np.str_('어린이(초등)'), np.float64(0.422)), (np.str_('소설'), np.float64(0.111)), (np.str_('인문'), np.float64(0.067))]



## 프롬프트 예시 (이 확인 셀)

```
'처음'이라는 단어가 학습 데이터에서 어느 분야 제목에 주로 쓰였는지 확인하고 싶습니다.
미분류를 뺀 데이터에서 상품명에 '처음'이 들어간 도서의 분야별 개수를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [8]:
# '처음'이라는 단어가 학습 데이터에서 어느 분야 제목에 들어 있었는가
train_df = df[df["분야"] != "미분류"]
train_df[train_df["상품명"].str.contains("처음")]["분야"].value_counts()

분야
어린이(초등)    7
인문         1
외국어        1
경제/경영      1
시/에세이      1
Name: count, dtype: int64

### 결과 해석

| 입력 | 사전에 있던 단어 | 예측 | 이유 |
|---|---|---|---|
| 처음 배우는 파이썬 데이터 분석 | `처음` 하나 | 어린이(초등) | `처음`이 들어간 학습 제목 11권 중 **7권이 어린이** |
| 인공지능 시대의 데이터 분석 | 없음 | 소설 | 단서가 0개라 가장 흔한 분야(소설)로 기움 |
| 처음 배우는 주식 투자 | 주식, 처음, 투자 | 경제/경영 | `주식`, `투자`가 경제 쪽 단서 |
| 2027 해커스 토익 기출 | 2027, 기출, 토익, 해커스 | 외국어 | `토익`이 결정적 (수험서와 0.309 vs 0.288로 근소) |
| 흔한남매 과학 탐험대 | 과학, 탐험대, 흔한남매 | 어린이(초등) | 시리즈명이 결정적 (확률 0.422) |

`파이썬`, `데이터`, `분석`, `인공지능`은 이번 베스트셀러 제목에 거의 없어서 **단어 사전에 없다.**
컴퓨터/IT 분야는 학습 데이터에 13권뿐이라 모델이 이 분야를 잘 모른다.

이 결과의 의미는 다음과 같다.

> 현재 학습 데이터의 제목 패턴을 바탕으로 모델이 해당 분야를 예측했습니다.

다음처럼 해석하지 않는다.

- 실제 공식 분야가 반드시 이 분야다. ×
- 모델이 제목의 의미를 완전히 이해했다. ×
- 예측이 100% 정답이다. ×

`처음 배우는 파이썬 데이터 분석`이 어린이로 나온 것은 **"처음"을 어린이 책의 단어로 배운 결과**일 뿐,
제목의 뜻을 이해한 것이 아니다.

---
# 실습 7. 추천용 TF-IDF 준비하기

## 해야 할 일

Chapter 05의 추천 기능을 앱에 연결한다.

분류용과 추천용 TF-IDF는 **목적이 다르다.**

| 용도 | 학습 대상 | 목적 |
|---|---|---|
| 분류용 | 미분류를 뺀 922권 | 분야 예측 |
| 추천용 | 전체 986권 | 도서 제목끼리 유사도 비교 |

추천은 **모든 도서가 선택 대상**이어야 하므로 미분류 도서도 포함한다.

### app.py 안의 코드

```python
@st.cache_resource
def build_recommender():
    df = load_data()
    vectorizer = TfidfVectorizer()
    title_matrix = vectorizer.fit_transform(df["상품명"])
    return vectorizer, title_matrix
```

## 프롬프트 예시

```
Streamlit 앱에서 쓸 추천용 TF-IDF를 준비하는 build_recommender() 함수를 만들고 싶습니다.
load_data()로 전체 도서를 불러와 상품명으로 TfidfVectorizer를 fit_transform하고
vectorizer와 제목 행렬 title_matrix를 돌려주세요.
st.cache_resource로 한 번만 만들게 해주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
app.py의 build_recommender()와 같은 로직을 Notebook에서 확인하고 싶습니다.
st.cache_resource만 빼고 같은 함수를 만들어 실행한 뒤,
추천용 행렬 크기와 분류용·추천용 단어 수를 비교해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [9]:
def build_recommender():
    df = load_data()
    vectorizer = TfidfVectorizer()
    title_matrix = vectorizer.fit_transform(df["상품명"])
    return vectorizer, title_matrix

_, title_matrix = build_recommender()

print("추천용 행렬 크기:", title_matrix.shape)
print("분류용 단어 수  :", len(classifier_vectorizer.vocabulary_))
print("추천용 단어 수  :", title_matrix.shape[1])

추천용 행렬 크기: (986, 2179)
분류용 단어 수  : 2144
추천용 단어 수  : 2179


---
# 실습 8. 도서 선택 UI 만들기

## 해야 할 일

사용자가 기존 도서 중 한 권을 선택하도록 한다.
**내부적으로는 index를 사용하고, 화면에는 사람이 읽기 쉬운 제목을 보여준다.**

### app.py 안의 코드

```python
def format_book(index):
    title = df.loc[index, "상품명"]
    if "저자" in df.columns:
        author = str(df.loc[index, "저자"])
        return f"{title} | {author}"
    return title

selected_index = st.selectbox(
    "기준 도서를 선택하세요",
    options=df.index.tolist(),
    format_func=format_book,
)
```

## 프롬프트 예시

```
Streamlit selectbox로 기준 도서를 고르게 하고 싶습니다.
선택지의 실제 값은 df의 index로 하고,
화면에는 '제목 | 저자' 형태로 보이도록 format_book(index) 함수를 만들어
format_func로 연결해 주세요. 저자 컬럼이 없으면 제목만 보여주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
selectbox에 쓸 format_book(index) 함수를 Notebook에서 확인하고 싶습니다.
'제목 | 저자' 형태로 돌려주는 함수를 만들고,
선택지 개수와 앞 5개가 화면에 어떻게 보이는지 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [10]:
def format_book(index):
    title = df.loc[index, "상품명"]
    if "저자" in df.columns:
        author = str(df.loc[index, "저자"])
        return f"{title} | {author}"
    return title

options = df.index.tolist()
print("선택지 개수:", len(options))
print()
for idx in options[:5]:
    print(f"  값 {idx:>3}  ->  화면 표시: {format_book(idx)}")

선택지 개수: 986

  값   0  ->  화면 표시: 세네카, 오늘을 빼앗기고 있는 당신에게 | 세네카
  값   1  ->  화면 표시: 흔한남매 23 | 흔한남매
  값   2  ->  화면 표시: 머니 트렌드 2027 | 정태익 외
  값   3  ->  화면 표시: 싯다르타 | 헤르만 헤세
  값   4  ->  화면 표시: 한국사 이상현상 연구원(일반판) | 최인서


같은 제목이 두 번 등록된 책도 화면에서는 똑같이 보이지만,
**실제 값(index)이 달라서** 서로 다른 선택지로 구분된다.

---
# 실습 9. Top 5 추천 연결하기

## 해야 할 일

Chapter 05에서 만든 코사인 유사도 추천을 함수로 연결한다.

```
도서 선택 -> 선택 도서와 전체 도서 유사도 -> 자기 자신 제외 -> 높은 순서 정렬 -> Top 5
```

이 함수는 자기 자신의 유사도를 **-1로 바꿔서** 정렬에서 맨 뒤로 보내는 방식으로 제외한다.

### app.py 안의 코드

```python
def recommend_books(df, title_matrix, selected_index, top_n=5):
    similarities = cosine_similarity(
        title_matrix[selected_index],
        title_matrix,
    ).ravel()
    similarities[selected_index] = -1
    top_indices = similarities.argsort()[::-1][:top_n]
    display_columns = [
        col for col in ["상품명", "저자", "출판사", "분야"] if col in df.columns
    ]
    result = df.iloc[top_indices][display_columns].copy()
    result["유사도"] = similarities[top_indices].round(3)
    return result

_, title_matrix = build_recommender()
st.header("2. 비슷한 도서 추천")
if st.button("비슷한 도서 5권 추천"):
    recommendations = recommend_books(df, title_matrix, selected_index, top_n=5)
    st.dataframe(recommendations, width="stretch", hide_index=True)
```

## 프롬프트 예시

```
Chapter 05의 코사인 유사도 추천을 Streamlit 앱용 함수로 만들고 싶습니다.
recommend_books(df, title_matrix, selected_index, top_n=5) 함수에서
선택 도서와 전체 도서의 유사도를 구하고,
자기 자신의 유사도는 -1로 바꿔 제외한 뒤 높은 순서로 top_n개를 고르고,
상품명·저자·출판사·분야(있는 것만)와 소수점 3자리 유사도를 표로 돌려주세요.
그리고 '비슷한 도서 5권 추천' 버튼을 누르면 이 결과를 st.dataframe으로 보여주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
app.py의 recommend_books()와 같은 로직을 Notebook에서 확인하고 싶습니다.
같은 함수를 만들고 0번, 1번 도서의 추천 결과를 표로 보여 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_books(df, title_matrix, selected_index, top_n=5):
    similarities = cosine_similarity(
        title_matrix[selected_index],
        title_matrix,
    ).ravel()
    similarities[selected_index] = -1
    top_indices = similarities.argsort()[::-1][:top_n]
    display_columns = [
        col for col in ["상품명", "저자", "출판사", "분야"] if col in df.columns
    ]
    result = df.iloc[top_indices][display_columns].copy()
    result["유사도"] = similarities[top_indices].round(3)
    return result

for idx in [0, 1]:
    print(f"[기준 {idx}] {df.loc[idx, '상품명']}")
    display(recommend_books(df, title_matrix, idx, top_n=5))

[기준 0] 세네카, 오늘을 빼앗기고 있는 당신에게


,상품명,저자,출판사,분야,유사도
587,"오늘을 빼앗기는 당신에게, 세네카",한주원,유페이퍼,미분류,0.668
554,돌이킬 수 있는,문목하,아작,소설,0.252
659,남아 있는 나날,가즈오 이시구로,민음사,소설,0.204
521,비전공자도 이해할 수 있는 LLM 수업,박상길,비즈니스북스,과학,0.162
349,품격 있는 대화를 위한 지식 브리핑,김진,북플레저,인문,0.147


[기준 1] 흔한남매 23


,상품명,저자,출판사,분야,유사도
542,흔한남매 22,흔한남매,미래엔아이세움,어린이(초등),0.408
344,흔한남매 이무기 4,흔한남매,미래엔아이세움,어린이(초등),0.395
719,흔한남매 세계사 탐험대 7,흔한남매,주니어김영사,어린이(초등),0.340
893,흔한남매 방방곡곡 한국사 1,흔한남매,미래엔아이세움,어린이(초등),0.332
741,흔한남매 과학 탐험대 17: 뇌와 호르몬,김언정,주니어김영사,어린이(초등),0.228


---
# 보충. 앱의 추천 함수는 Chapter 05와 무엇이 다른가

## 해야 할 일

Chapter 05의 `recommend_books()`는 **자기 자신과 같은 제목도** 제외했다.
Chapter 06의 앱 버전은 **자기 자신(같은 index)만** 제외한다.

Chapter 05에서 같은 책이 `미분류`와 실제 분야로 **두 번 등록**된 경우를 확인했다.
이 차이가 앱에서 어떻게 나타나는지 확인한다.

## 프롬프트 예시 (이 확인 셀)

```
같은 제목이 두 번 등록된 5번 도서('테오')를 기준으로
앱의 recommend_books로 추천 결과를 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [12]:
# 같은 제목이 두 번 등록된 '테오'를 기준으로 추천해 본다
teo_index = 5
print(f"[기준 {teo_index}] {df.loc[teo_index, '상품명']}  ({df.loc[teo_index, '분야']})")
display(recommend_books(df, title_matrix, teo_index, top_n=5))

[기준 5] 테오  (소설)


,상품명,저자,출판사,분야,유사도
221,테오,앨런 레비 외,오팬하우스,미분류,1.0
975,인간의 굴레에서 2,서머싯 몸,민음사,소설,0.0
0,"세네카, 오늘을 빼앗기고 있는 당신에게",세네카,논픽션,인문,0.0
985,세종의 나라 2,김진명,이타북스,미분류,0.0
984,교보문고 북커버 키트(브라운),교보문고,교보문고,교보문고 굿즈,0.0


## 프롬프트 예시 (이 확인 셀)

```
앱의 추천 방식(자기 자신만 제외)으로 986권 전체를 확인하고 싶습니다.
전체 유사도 행렬에서 대각선을 -1로 바꾸고 각 도서의 1위 추천을 구한 뒤,
1위가 자기와 같은 제목의 복제본인 책 수와
1위 유사도가 0 이하인 책 수를 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [13]:
# 전체 도서 중 앱 추천 1위가 '자기와 같은 제목의 복제본'인 책은 몇 권인가
import numpy as np

all_sim = cosine_similarity(title_matrix)
np.fill_diagonal(all_sim, -1)             # 앱과 같이 자기 자신만 제외
top1 = all_sim.argmax(axis=1)

same_title_top1 = sum(
    1 for i, j in enumerate(top1)
    if df.loc[i, "상품명"] == df.loc[j, "상품명"]
)
print("추천 1위가 같은 제목의 복제본인 책:", same_title_top1, "권")
print("추천 1위 유사도가 0 이하인 책    :", int((all_sim.max(axis=1) <= 0).sum()), "권")

추천 1위가 같은 제목의 복제본인 책: 87 권
추천 1위 유사도가 0 이하인 책    : 228 권


### 결과

`테오`를 고르면 추천 1위가 **`테오`(유사도 1.0)** 로 나온다.
같은 책이 `미분류`로 한 번 더 등록되어 있어서, 앱 입장에서는 "다른 책"이기 때문이다.
그 아래 2~5위는 유사도가 모두 0이라 의미 없는 추천이다.

전체 986권 중 **87권**이 추천 1위로 자기 자신의 복제본을 받는다.

앱은 블로그 코드 그대로 두었지만, 실제 서비스라면
Chapter 05처럼 **같은 제목을 제외하는 조건**을 추가하는 것이 좋다.

---
# 실습 10. 최종 app.py 통합하기

## 해야 할 일

지금까지의 코드를 하나의 `app.py`로 합친다. 코드를 한 줄씩 외우지 않고 **역할만 구분**한다.

| 부분 | 역할 |
|---|---|
| `load_data()` | 데이터 |
| `train_classifier()` | 분야 분류 |
| `build_recommender()` | 추천용 벡터 |
| `recommend_books()` | Top 5 추천 |
| Streamlit UI | 사용자 입력과 결과 출력 |

앱 시작 부분은 `try/except`로 감싸서, CSV가 없거나 컬럼이 없을 때
**빨간 오류 메시지를 화면에 보여주고 멈추게** 했다(`st.error`, `st.stop`).

완성된 코드는 같은 폴더의 `app.py`에 있다. 아래 셀로 내용을 확인할 수 있다.

## 프롬프트 예시

```
지금까지 만든 load_data, train_classifier, build_recommender, recommend_books와
분야 예측 화면, 도서 추천 화면을 하나의 app.py로 합치고 싶습니다.
페이지 제목은 '베스트셀러 텍스트 분석 앱', 레이아웃은 wide로 하고,
데이터·모델 준비 단계에서 FileNotFoundError나 ValueError가 나면
st.error로 보여주고 st.stop()으로 멈춰 주세요.
두 기능 사이에는 구분선을 넣고,
맨 아래에 분류·추천 결과의 한계를 알리는 안내 문구를 st.caption으로 넣어 주세요.
함수 이름과 순서는 위에서 말한 그대로 유지해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

## 프롬프트 예시 (이 확인 셀)

```
완성한 app.py에 필요한 부분이 다 들어 있는지 확인하고 싶습니다.
파일을 읽어 줄 수를 출력하고,
load_data, train_classifier, build_recommender, recommend_books 함수와
text_input, selectbox, dataframe이 포함되어 있는지 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [14]:
app_code = Path("app.py").read_text(encoding="utf-8")

print("app.py 줄 수:", len(app_code.splitlines()))
print()
for name in ["def load_data", "def train_classifier", "def build_recommender",
             "def recommend_books", "st.text_input", "st.selectbox", "st.dataframe"]:
    print(f"  {name:22} 포함: {name in app_code}")

app.py 줄 수: 164

  def load_data          포함: True
  def train_classifier   포함: True
  def build_recommender  포함: True
  def recommend_books    포함: True
  st.text_input          포함: True
  st.selectbox           포함: True
  st.dataframe           포함: True


---
# 실습 11. 최종 앱 실행 및 검증

## 해야 할 일

터미널에서 앱을 실행한다.

```
cd notebooks/book-text-ml
python -m streamlit run app.py
```

### 브라우저에서 직접 확인할 것

**분류 기능** — 서로 다른 제목을 2~3개 입력한다.
- 빈 제목 -> 안내 메시지
- 정상 제목 -> 예상 분야 출력

**추천 기능** — 기준 도서를 2~3권 바꿔 본다.
- 자기 자신이 제외되는가?
- 5권이 출력되는가?
- 유사도가 높은 순서인가?
- 실제 제목을 보았을 때 결과를 설명할 수 있는가?

> **앱이 실행된다는 것과 분석 결과가 적절하다는 것은 다른 문제다.**

### 자동으로 확인하기 — AppTest

Streamlit에는 브라우저 없이 앱을 실행하고 **버튼 클릭과 입력을 흉내 내는** `AppTest` 기능이 있다.
아래 셀들은 실제 `app.py`를 그대로 실행해서 위 체크리스트를 자동으로 확인한다.

## 프롬프트 예시 (이 확인 셀)

```
브라우저 없이 app.py를 자동으로 실행해서 검증하고 싶습니다.
streamlit.testing.v1의 AppTest로 app.py를 실행하고,
오류 여부, 제목, 헤더, 버튼 이름, 안내 문구 개수를 출력해 주세요.
Notebook에서 나오는 Streamlit 경고 로그는 숨겨 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [15]:
import logging
from streamlit.testing.v1 import AppTest

# Notebook에서 Streamlit을 돌릴 때 나오는 경고 메시지를 숨긴다
for name in list(logging.root.manager.loggerDict):
    if name.startswith("streamlit"):
        logging.getLogger(name).setLevel(logging.ERROR)

at = AppTest.from_file(str(Path("app.py").resolve()), default_timeout=60).run()

print("앱 실행 중 오류:", at.exception or "없음")
print("제목  :", at.title[0].value)
print("헤더  :", [h.value for h in at.header])
print("버튼  :", [b.label for b in at.button])
print("안내문:", len(at.caption), "개")

앱 실행 중 오류: 없음
제목  : 교보문고 베스트셀러 텍스트 분석 앱
헤더  : ['1. 도서 분야 예측', '2. 비슷한 도서 추천']
버튼  : ['분야 예측', '비슷한 도서 5권 추천']
안내문: 2 개


## 프롬프트 예시 (이 확인 셀)

```
AppTest 객체 at이 있습니다.
제목 입력창에 빈 값과 제목 3개를 차례로 넣고 '분야 예측' 버튼을 눌러
경고 메시지와 예측 결과가 제대로 나오는지 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [16]:
# 분류 기능 — 빈 제목과 정상 제목을 넣고 버튼을 눌러 본다
at.text_input[0].input("").run()
at.button[0].click().run()
print("빈 제목   ->", [w.value for w in at.warning])

for title in ["처음 배우는 주식 투자", "2027 해커스 토익 기출", "흔한남매 과학 탐험대"]:
    at.text_input[0].input(title).run()
    at.button[0].click().run()
    print(f"{title:18} ->", [s.value for s in at.success])

빈 제목   -> ['도서 제목을 입력해 주세요.']
처음 배우는 주식 투자       -> ['예상 분야: 경제/경영']
2027 해커스 토익 기출     -> ['예상 분야: 외국어']


흔한남매 과학 탐험대        -> ['예상 분야: 어린이(초등)']


## 프롬프트 예시 (이 확인 셀)

```
AppTest 객체 at이 있습니다.
기준 도서를 0, 1, 8번으로 바꿔 가며 '비슷한 도서 5권 추천' 버튼을 누르고,
결과가 5권인지, 유사도가 내림차순인지, 추천 1위가 무엇인지 출력해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [17]:
# 추천 기능 — 기준 도서를 바꿔 가며 체크리스트를 확인한다
for idx in [0, 1, 8]:
    at.selectbox[0].set_value(idx).run()
    at.button[1].click().run()
    table = at.dataframe[0].value
    selected = df.loc[idx, "상품명"]

    print(f"[기준 {idx}] {selected}")
    print("   5권인가?          :", len(table) == 5)
    print("   유사도 내림차순인가?:", table["유사도"].is_monotonic_decreasing)
    print("   추천 1위           :", table.iloc[0]["상품명"], f"({table.iloc[0]['유사도']})")
    print()

[기준 0] 세네카, 오늘을 빼앗기고 있는 당신에게


   5권인가?          : True
   유사도 내림차순인가?: True
   추천 1위           : 오늘을 빼앗기는 당신에게, 세네카 (0.668)

[기준 1] 흔한남매 23
   5권인가?          : True
   유사도 내림차순인가?: True
   추천 1위           : 흔한남매 22 (0.408)

[기준 8] 판매의 법칙
   5권인가?          : True
   유사도 내림차순인가?: True
   추천 1위           : 불변의 법칙 (0.43)



### 검증 결과

| 확인 항목 | 결과 |
|---|---|
| 앱 실행 중 오류 | 없음 |
| 빈 제목 입력 | `도서 제목을 입력해 주세요.` 안내 |
| 정상 제목 | 예상 분야 출력 (주식 투자 -> 경제/경영, 토익 -> 외국어, 흔한남매 -> 어린이) |
| 추천 개수 | 5권 |
| 유사도 순서 | 내림차순 |

앱의 동작 자체는 모두 정상이다. 하지만 실습 6과 보충에서 확인했듯이
**동작이 정상이라고 결과가 적절한 것은 아니다.**

---
# 실습 12. 주요 오류 빠르게 확인하기

### CSV를 찾지 못함 — `FileNotFoundError`
- `app.py`와 CSV가 같은 폴더인가?
- 파일명이 정확한가?
- **현재 작업 폴더가 맞는가?** <- 이 저장소에서는 이게 가장 흔한 원인이다.
  터미널이 `C:\dev\llm-data-analysis-course`에 있으면 실패한다.
  `cd notebooks/book-text-ml` 후에 실행한다.

PowerShell에서 현재 폴더의 파일을 확인한다.
```
dir
```

### 필수 컬럼 없음
실제 컬럼을 확인한다.

### TF-IDF vocabulary 오류
상품명 데이터가 비어 있지 않은지 확인한다.

### 예측 또는 추천 결과가 이상함
바로 알고리즘 오류라고 판단하지 않는다.

```
입력 확인 -> 실제 데이터 확인 -> index 확인 -> 학습 데이터/분야 분포 확인 -> 모델 또는 추천의 한계 확인
```

## 프롬프트 예시 (이 확인 셀)

```
앱에서 자주 나는 오류를 미리 점검하고 싶습니다.
현재 작업 폴더, CSV 존재 여부, 실제 컬럼 목록,
상품명 앞 5개, 빈 상품명 개수를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [18]:
# 위 오류들을 미리 점검한다
print("현재 작업 폴더:", Path.cwd())
print("CSV 있음:", Path(DATA_PATH).exists())
print()
print("실제 컬럼:", df.columns.tolist())
print()
print("상품명 앞 5개:", df["상품명"].head().tolist())
print("빈 상품명 수 :", int((df["상품명"].str.len() == 0).sum()))

현재 작업 폴더: C:\dev\llm-data-analysis-course\notebooks\book-text-ml
CSV 있음: True

실제 컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']

상품명 앞 5개: ['세네카, 오늘을 빼앗기고 있는 당신에게', '흔한남매 23', '머니 트렌드 2027', '싯다르타', '한국사 이상현상 연구원(일반판)']
빈 상품명 수 : 0


---
# 실습 13. 전체 프로젝트 마무리

```
Chapter 01   데이터 전처리
    ↓
Chapter 02   단어 빈도와 Word Cloud
    ↓
Chapter 03   CountVectorizer와 TF-IDF
    ↓
Chapter 04   Naive Bayes 분야 분류
    ↓
Chapter 05   Cosine Similarity 유사 도서 추천
    ↓
Chapter 06   Streamlit 분류·추천 앱
```

최종적으로 다음 전체 과정을 경험했다.

> 데이터 -> 전처리 -> 텍스트 분석 -> 수치화 -> 머신러닝 -> 유사도 추천 -> 사용자 앱

### 여섯 Chapter를 관통한 발견

| Chapter | 발견 | 이후에 미친 영향 |
|---|---|---|
| 01 | 도서가 아닌 상품(디퓨저, 기프트카드)이 섞여 있음 | 02 단어 집계, 04 오분류, 05 추천에 계속 등장 |
| 01 | 분야 결측 64권을 `미분류`로 채움 | 04에서 정답에서 제외, 05·06에서 **같은 책 복제본**으로 추천 1위 |
| 02 | 수험서는 전용 단어(해커스, 기출)가 반복됨 | 04에서 수험서·외국어만 F1 0.86 |
| 03 | 기본 TF-IDF가 조사를 떼지 않고 연도를 단어로 봄 | 04 오분류 25건, 05 연도 기반 엉뚱한 추천 |
| 04 | 단서가 없으면 가장 흔한 분야(소설)로 예측 | 06 앱에서 `인공지능 시대의 데이터 분석 -> 소설` |
| 05 | 26.5%의 책은 겹치는 단어가 없어 추천 근거가 없음 | 06 앱에서도 유사도 0인 추천이 그대로 나옴 |

> **Chapter 01의 데이터 문제가 Chapter 06의 앱 화면까지 그대로 이어진다.**

---
# 결과 정리

> 아래 숫자는 위 셀들의 **실제 실행 결과**를 보고 적은 것이다.

## Chapter 06 결과

### 앱 구성
- `app.py` 한 파일, 실행: `cd notebooks/book-text-ml` -> `python -m streamlit run app.py`
- 분류 모델: 미분류를 뺀 **922권, 37개 분야**로 학습한 TF-IDF + MultinomialNB
- 추천: 전체 **986권**의 제목 TF-IDF 코사인 유사도, 자기 자신 제외 Top 5
- 캐시: `load_data()`는 `st.cache_data`, 모델·행렬은 `st.cache_resource`

### 동작 검증 (AppTest)
- 앱 실행 오류 없음
- 빈 제목 -> 안내 메시지, 정상 제목 -> 예상 분야
- 추천 5권, 유사도 내림차순

### 결과 검증
| 입력 | 예측 | 판단 |
|---|---|---|
| 처음 배우는 주식 투자 | 경제/경영 | 적절 (`주식`, `투자`) |
| 2027 해커스 토익 기출 | 외국어 | 적절하나 수험서와 근소 (0.309 vs 0.288) |
| 흔한남매 과학 탐험대 | 어린이(초등) | 적절 (시리즈명) |
| 처음 배우는 파이썬 데이터 분석 | 어린이(초등) | **부적절** — `처음`만 사전에 있었고, 학습 데이터에서 주로 어린이 책에 쓰인 단어 |
| 인공지능 시대의 데이터 분석 | 소설 | **부적절** — 사전 단어 0개, 가장 흔한 분야로 기움 |

추천은 시리즈(`흔한남매 23 -> 흔한남매 5권`)는 잘 찾지만,
`테오`처럼 같은 책이 두 번 등록된 경우 **자기 복제본이 1위(유사도 1.0)** 로 나온다 (전체 87권).

---
# 핵심 시사점

### 1. 앱이 돌아간다는 것과 결과가 맞다는 것은 다르다
AppTest의 모든 동작 검사는 통과했다. 하지만 `처음 배우는 파이썬 데이터 분석`은 어린이,
`인공지능 시대의 데이터 분석`은 소설로 나왔다. **화면 검증과 결과 검증은 따로 해야 한다.**

### 2. 사용자는 학습 데이터에 없는 제목을 입력한다
분석할 때는 데이터 안의 제목만 봤지만, 앱 사용자는 `파이썬`, `인공지능`처럼
**학습 데이터에 없는 단어**로 제목을 입력한다. 이런 입력에서 모델의 약점이 가장 먼저 드러난다.

### 3. 모델이 "모른다"고 말하지 못한다
사전 단어가 0개여도 앱은 자신 있게 `예상 분야: 소설`이라고 보여준다.
실제 서비스라면 **확률이 낮거나 사전 단어가 없을 때 "판단하기 어렵다"고 안내**하는 것이 정직하다.

### 4. 데이터 정리 결과가 사용자 화면까지 간다
Chapter 01에서 `미분류`로 채운 복제 도서가 추천 1위로 나오고, 디퓨저·화장지가 추천 목록에 섞인다.
**전처리 단계의 판단이 최종 사용자 경험을 결정한다.**

### 5. 캐시는 앱을 쓸 만하게 만드는 장치다
Streamlit은 버튼을 누를 때마다 전체 코드를 다시 실행한다.
캐시가 없으면 매번 CSV 로드와 모델 학습이 반복된다.

## 개선 방향

| 방향 | 기대 효과 |
|---|---|
| 예측 확률이나 사전 단어 수가 낮으면 "판단 어려움" 표시 | 근거 없는 예측을 사용자에게 알림 |
| 예측과 함께 확률 상위 3개 분야를 보여주기 | 수험서/외국어처럼 근소한 경우를 드러냄 |
| 추천에서 같은 제목 제외 (Chapter 05 방식) | 복제본 1위 문제 해결 (87권) |
| 추천 유사도가 0이면 "비슷한 도서 없음" 표시 | 화장지·북커버 같은 무의미한 추천 제거 |
| 비도서 상품과 중복 등록 데이터 정리 | 분류와 추천 모두 개선 |
| Kiwi 형태소 분석으로 명사만 사용 | `파이썬으로`, `부자로` 같은 조사 문제 해결 |

---
# 최종 체크리스트

- [x] Streamlit 앱을 실행했다.
- [x] CSV가 정상적으로 로드된다.
- [x] 도서 제목 입력창이 보인다.
- [x] 빈 제목 입력을 처리한다.
- [x] 새 제목은 기존 Vectorizer의 `transform()`을 사용한다.
- [x] 예상 분야가 화면에 출력된다.
- [x] 기존 도서를 선택할 수 있다.
- [x] 자기 자신을 제외한 유사 도서 5권이 출력된다.
- [x] 유사도 순서를 확인했다.
- [x] 서로 다른 입력으로 분류와 추천을 직접 검증했다.
- [x] 분류와 추천 결과의 한계를 설명할 수 있다.

---
# 이번 Chapter에서 꼭 기억할 5가지

1. Chapter 06은 새로운 알고리즘보다 **기존 분석 기능을 UI에 연결**하는 단계다.
2. 새 제목 예측에는 기존 Vectorizer의 **`transform()`** 을 사용한다.
3. 캐시는 반복되는 데이터 로딩과 모델 생성을 줄이는 데 사용한다.
4. 추천 결과는 제목 TF-IDF 유사도이며 **개인 취향 추천은 아니다.**
5. 화면이 정상적으로 보여도 **실제 분류와 추천 결과를 직접 검증**해야 한다.

## Chapter 06 한 문장 정리

> 앞에서 만든 TF-IDF 분류와 코사인 유사도 추천 기능을 Streamlit의 입력·선택 UI에 연결하고,
> 실제 사용자의 관점에서 결과를 검증하여 하나의 데이터 분석 앱으로 완성한다.